# RE:ARCHIVE — 무료 / 고품질 정면 원화 인계
원화 S01(1536×2304 RGBA)가 승인된 뒤 실행. G1 별도 좌우 45° 레퍼런스는 자동 분리 입력이 아닙니다. **이 노트북은 그림·리깅을 자동 제작하지 않고 실제 입력을 검증합니다.**

In [ ]:
from pathlib import Path
import subprocess,sys,uuid,json
REPO=Path("/content/Virtual-pipeline")
URL="https://github.com/jujumelona/Virtual-pipeline.git"
if (REPO/".git").exists():
    subprocess.run(["git","-C",str(REPO),"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","-B","main","origin/main"],check=True)
else:
    subprocess.run(["git","clone","--depth","1","--branch","main",URL,str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","Pillow"],check=True)
CONTRACT=REPO/"docs/REARCHIVE_PRODUCTION_CONTRACT.json"
CHECK=REPO/"tools/rearchive_production_preflight.py"
assert CONTRACT.is_file() and CHECK.is_file()

In [ ]:
from google.colab import files
WORK=Path("/content")/("rearchive_"+uuid.uuid4().hex[:8]);WORK.mkdir()
print("REARCHIVE_ART_SOURCE_INPUT.zip 안에 art/S01_front_neutral.png 가 있어야 합니다.")
u=files.upload()
if len(u)!=1:raise RuntimeError("ZIP 하나만 업로드")
fn,raw=next(iter(u.items()))
if not fn.lower().endswith(".zip"):raise RuntimeError("ZIP만 허용")
ART=WORK/"REARCHIVE_ART_SOURCE_INPUT.zip";ART.write_bytes(raw)

In [ ]:
STAGE="front" #@param ["front","sheets","parts"]
REPORT=WORK/("qa_"+STAGE+".json")
subprocess.run([sys.executable,str(CHECK),"--input",str(ART),
                "--contract",str(CONTRACT),"--stage",STAGE,
                "--report",str(REPORT)],check=True)
print(REPORT.read_text()[:7000])
print("검증은 파일 형식/좌표만 확인했습니다. 얼굴 동일성/아트 품질은 별도 검수 필요")

In [ ]:
#@title 기존 공식 See-through용 ZIP 인계
if STAGE!="front":raise RuntimeError("원화 1장 입력은 front로 재검사")
PACKAGE=WORK/"complete_character__ARTWORK.zip"
subprocess.run([sys.executable,str(CHECK),"--input",str(ART),
                "--contract",str(CONTRACT),"--stage","front",
                "--report",str(WORK/"qa_front.json"),
                "--prepare-seethrough-zip",str(PACKAGE)],check=True)
files.download(str(PACKAGE))
print("GPU 분리는 다음 기존 노트북에서 수행: https://colab.research.google.com/github/jujumelona/Virtual-pipeline/blob/main/notebooks/FACE_SeeThrough_OfficialAligned_v3.ipynb")
print("PSD/리깅 생성 완료로 표시하지 마세요.")